# Probably High-Value? — Viva Presentation Walkthrough

This thin notebook calls modules in `src/` and displays the key artifacts from `reports/tables/` and `reports/figures/` for the **ABAI2009L (Data Management and AI Integration)** viva presentation.

## System Architecture

![System Architecture](../docs/architecture.png)

- **Topic 3 (Core — KDD & Data Mining):** 6-step SQL cleaning funnel, temporal snapshots (`S1`, `S2`, `S3`), `spherical, K=5` GMM segmentation, depth-4 decision-tree surrogate, and calibrated 90-day lapse model.
- **Topic 1 (Probabilistic Databases):** Block-Independent-Disjoint (`segment_membership`) and tuple-independent (`lapse_risk`) tables queried via parameterised templates `Q1`–`Q6`.
- **Topic 2 (NLP for Database Interaction):** Deterministic `spaCy` + IPCC/Kent hedge lexicon parser mapping plain-English questions to safe SQL parameters.


In [1]:
import sys
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import config, db, prob_queries
from src.nl.engine import answer_question

## 1. Data Cleaning Funnel & Snapshot Summary

In [2]:
display(pd.read_csv(config.TABLES_DIR / "cleaning_funnel.csv"))
display(pd.read_csv(config.TABLES_DIR / "snapshot_summary.csv"))

,step_no,rule,rows_before,rows_after,rows_removed,revenue_affected,share_of_revenue_before_step_pct,share_of_raw_revenue_pct
0,1,Overlap duplicates between sheets,1067371,1044848,22523,377488.45,1.96,1.96
1,2,Remaining exact duplicates,1044848,1033036,11812,54228.42,0.29,0.28
2,3,Cancellations moved to returns,1033036,1013932,19104,-1462050.61,-7.75,-7.58
3,4,Quantity or price not positive,1013932,1007913,6019,-158676.14,-0.78,-0.82
4,5,Non-product stock codes,1007913,1003357,4556,832398.81,4.07,4.32
5,6,Missing customer_id (excluded from customer analysis),1003357,776596,226761,2575278.90,13.11,13.35


,snapshot,snapshot_date,feature_start,feature_end,outcome_start,outcome_end,n_customers,mean_recency_days,median_recency_days,mean_frequency,median_frequency,mean_monetary,median_monetary,mean_tenure_days,median_tenure_days,mean_avg_basket,median_avg_basket,n_purchased_next_90d,n_lapsed_next_90d,purchase_rate_pct,lapse_rate_pct
0,S1,2010-09-10,2009-12-10,2010-09-09,2010-09-10,2010-12-09,3256,87.89,72.0,3.53,2.0,1598.59,585.77,181.07,192.0,366.80,287.52,1908.0,1348.0,58.60,41.40
1,S2,2011-09-10,2010-12-10,2011-09-09,2011-09-10,2011-12-09,3247,88.36,69.0,3.40,2.0,1555.65,552.00,405.23,460.0,401.43,283.12,1871.0,1376.0,57.62,42.38
2,S3,2011-12-10,2011-03-10,2011-12-09,NaN,NaN,3961,71.38,43.0,3.70,2.0,1768.34,631.89,436.50,478.0,408.99,296.64,NaN,NaN,NaN,NaN


## 2. S3 Segment Profiles, Gate Report & Hard vs Expected Sizes

In [3]:
display(pd.read_csv(config.TABLES_DIR / "segment_profiles_spherical_k5.csv"))
display(pd.read_csv(config.TABLES_DIR / "gate.csv"))
display(pd.read_csv(config.TABLES_DIR / "kmeans_vs_expected_sizes.csv"))

,cluster_id,n_customers,mean_recency_days,median_recency_days,mean_frequency,median_frequency,mean_monetary,median_monetary,mean_tenure_days,mean_avg_basket,ave_pp,share_of_customers_pct,jaccard_mean,bootstrap_ari_median,bootstrap_ari_iqr,proposed_segment
0,4,258,8.60,5.0,18.66,14.0,12635.95,5455.26,620.17,620.79,0.8934,6.51,0.8669,0.8674,0.0977,Champions
1,0,1105,21.72,17.0,5.04,5.0,2154.78,1579.14,478.98,530.84,0.8490,27.90,0.8775,0.8674,0.0977,Loyal
2,1,933,86.66,72.0,2.36,2.0,978.14,786.57,447.68,517.56,0.7880,23.55,0.8449,0.8674,0.0977,At-Risk
3,2,636,23.19,22.0,1.42,1.0,301.97,284.55,320.97,234.11,0.8086,16.06,0.8758,0.8674,0.0977,New
4,3,1029,156.38,159.0,1.14,1.0,251.34,229.23,406.09,234.67,0.8723,25.98,0.9009,0.8674,0.0977,Hibernating


,method,n_customers,n_boundary_max_p_lt_0_6,boundary_share_pct,mean_max_p,median_max_p,mean_normalised_entropy,median_normalised_entropy,ave_pp_Champions,ave_pp_Loyal,ave_pp_At-Risk,ave_pp_New,ave_pp_Hibernating
0,gmm,3961,495,12.50,0.8371,0.8996,0.2526,0.2253,0.8934,0.8490,0.7880,0.8086,0.8723
1,bootstrap,3961,152,3.84,0.9329,0.9900,0.1041,0.0348,0.9490,0.9339,0.9191,0.9409,0.9352


,segment,kmeans_hard_count,gmm_hard_count,gmm_expected_size,gmm_std_dev,bootstrap_hard_count,bootstrap_expected_size,bootstrap_std_dev
0,Champions,364,258,278.63,7.43,278,290.55,5.63
1,Loyal,887,1105,1073.79,14.71,1116,1110.32,10.22
2,At-Risk,901,933,943.04,16.99,928,941.95,10.91
3,New,736,636,670.19,14.07,634,643.37,7.81
4,Hibernating,1073,1029,995.35,12.79,1005,974.81,8.60


## 3. Held-Out S2 90-Day Lapse Evaluation

In [4]:
display(pd.read_csv(config.TABLES_DIR / "lapse_s2_eval.csv"))

,split,model,n_customers,base_lapse_rate,mean_predicted_p,brier_score,brier_skill_score,roc_auc,log_loss,ece_10_bins
0,S2_heldout_test,logistic_calibrated,3247,0.4238,0.4008,0.2034,0.1672,0.7390,0.5870,0.0343
1,S2_heldout_test,tree_depth4_calibrated,3247,0.4238,0.3801,0.2143,0.1225,0.7106,0.6145,0.0774
2,S2_heldout_test,two_bin_recency_90d,3247,0.4238,0.4125,0.2264,0.0730,0.6371,0.6448,0.0206


## 4. Live Probabilistic Query & Natural-Language Engine Demo (Read-Only `app.duckdb`)

In [5]:
with db.connect_app() as con:
    response = answer_question(
        con,
        "Which customers in Germany are probably Champions and likely to lapse?",
        default_method="gmm",
    )
print("Interpretation:", response.parsed.interpretation)
print("Template:", response.parsed.template_id, "| Params:", response.parsed.params)
display(response.rows)


Interpretation: Showing customers in Germany with P(Champions) ≥ 0.66 ('probably') and P(lapse) ≥ 0.66 ('likely') (method = 'gmm').
Template: Q6 | Params: {'method': 'gmm', 'seg': 'Champions', 'tau1': 0.66, 'max_p1': 1.0, 'only_top_segment': False, 'tau2': 0.66, 'max_p2': 1.0, 'country': 'Germany', 'limit': 500}


,customer_id,country,segment,p,p_lapse_90d,recency_days,frequency,monetary
